# Performance Optimization

Master GPU performance analysis through memory coalescing, the roofline model for identifying bottlenecks, and systematic profiling techniques to iteratively optimize your CUDA kernels.

[Open this lesson on the site](https://llm.thelittleone.rocks/#/module/cuda-performance)


## Runtime setup

Before running, set **Runtime -> Change runtime type -> T4 GPU** (or any available GPU). Then run the install cell below.

In [ ]:
!pip install numba

---

## Lesson examples (optional)

These are the code samples from the lesson sections. Run them to experiment with the concepts before tackling the exercise below.

### Lesson example: Memory Coalescing Deep Dive

In [ ]:
!pip install numba

from numba import cuda
import numpy as np
import math
import time

# ------------------------------------------------------------------
# Experiment: Coalesced vs Strided Memory Access
# ------------------------------------------------------------------

# Kernel: coalesced row-wise read of a 2D matrix
@cuda.jit
def read_coalesced(matrix, output, rows, cols):
    """Each thread reads one element; consecutive threads read consecutive columns."""
    row = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
    col = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    if row < rows and col < cols:
        # Consecutive threads (varying threadIdx.x) read consecutive cols
        # -> consecutive memory addresses -> COALESCED
        output[row * cols + col] = matrix[row, col] * 2.0

# Kernel: uncoalesced column-wise read of a 2D matrix
@cuda.jit
def read_strided(matrix, output, rows, cols):
    """Each thread reads one element; consecutive threads read consecutive rows."""
    col = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
    row = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    if row < rows and col < cols:
        # Consecutive threads (varying threadIdx.x) read consecutive rows
        # -> addresses are 'cols' apart -> STRIDED -> UNCOALESCED
        output[row * cols + col] = matrix[row, col] * 2.0

def bench_kernel(kernel, args, grid, block, warmup=10, iters=50):
    for _ in range(warmup):
        kernel[grid, block](*args)
    cuda.synchronize()
    start = time.perf_counter()
    for _ in range(iters):
        kernel[grid, block](*args)
    cuda.synchronize()
    return (time.perf_counter() - start) / iters * 1000  # ms

# Setup: large matrix
rows, cols = 4096, 4096
matrix = np.random.randn(rows, cols).astype(np.float32)
d_matrix = cuda.to_device(matrix)
d_output = cuda.device_array(rows * cols, dtype=np.float32)

# Block and grid for coalesced (x = col, y = row)
BLOCK = (32, 8)
grid_coal = (math.ceil(cols / BLOCK[0]), math.ceil(rows / BLOCK[1]))

# Block and grid for strided (x = row, y = col)
grid_stri = (math.ceil(rows / BLOCK[0]), math.ceil(cols / BLOCK[1]))

data_bytes = rows * cols * 4  # float32

t_coal = bench_kernel(read_coalesced, (d_matrix, d_output, rows, cols), grid_coal, BLOCK)
t_stri = bench_kernel(read_strided, (d_matrix, d_output, rows, cols), grid_stri, BLOCK)

bw_coal = (data_bytes * 2) / (t_coal / 1000) / 1e9  # GB/s (read + write)
bw_stri = (data_bytes * 2) / (t_stri / 1000) / 1e9

print("=== Memory Coalescing Experiment ===")
print(f"Matrix: {rows} x {cols} float32 ({data_bytes / 1e6:.1f} MB)\n")
print(f"{'Pattern':<25} {'Time (ms)':<14} {'Bandwidth (GB/s)':<20} {'Relative'}")
print("-" * 70)
print(f"{'Coalesced (row-wise)':<25} {t_coal:<14.3f} {bw_coal:<20.1f} {'1.00x (baseline)'}")
print(f"{'Strided (col-wise)':<25} {t_stri:<14.3f} {bw_stri:<20.1f} {t_stri/t_coal:.1f}x slower")
print(f"\nBandwidth utilization:")
print(f"  Coalesced: {bw_coal:.1f} GB/s")
print(f"  Strided:   {bw_stri:.1f} GB/s")
print(f"  Efficiency ratio: {bw_stri/bw_coal*100:.1f}%")

# ------------------------------------------------------------------
# Experiment 2: Stride size impact
# ------------------------------------------------------------------
@cuda.jit
def read_with_stride(data, output, n, stride):
    idx = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if idx < n:
        src_idx = (idx * stride) % n  # Wrap around
        output[idx] = data[src_idx] * 2.0

n = 2_000_000
d_data = cuda.to_device(np.random.randn(n).astype(np.float32))
d_out = cuda.device_array(n, dtype=np.float32)
TPB = 256
BPG = math.ceil(n / TPB)

print("\n=== Stride Size vs Performance ===")
print(f"{'Stride':<10} {'Time (ms)':<14} {'Effective BW (GB/s)':<22} {'Slowdown'}")
print("-" * 60)

baseline = None
for stride in [1, 2, 4, 8, 16, 32, 64, 128]:
    t = bench_kernel(read_with_stride, (d_data, d_out, n, stride), BPG, TPB)
    bw = (n * 4 * 2) / (t / 1000) / 1e9
    if baseline is None:
        baseline = t
    label = "(coalesced)" if stride == 1 else ""
    print(f"{stride:<10} {t:<14.4f} {bw:<22.1f} {t/baseline:.2f}x {label}")

### Lesson example: Arithmetic Intensity & Roofline Model

In [ ]:
!pip install numba

from numba import cuda
import numpy as np
import math
import time

# ------------------------------------------------------------------
# Demo 1: Calculate arithmetic intensity for common kernels
# ------------------------------------------------------------------
print("=== Arithmetic Intensity of Common Kernels ===")
print(f"{'Kernel':<25} {'FLOPS/elem':<12} {'Bytes/elem':<12} {'AI (F/B)':<10} {'Bound'}")
print("-" * 72)

kernels = [
    ("Vector add (a+b)",       1,  12, "Memory"),
    ("SAXPY (a*x+y)",          2,  12, "Memory"),
    ("Vector norm (x*x)",      1,   8, "Memory"),
    ("5-pt stencil",           5,  24, "Memory"),
    ("7x7 Convolution",       98,   4, "Balanced"),
    ("MatMul 256x256",       512,  12, "Compute"),
    ("MatMul 1024x1024",    2048,  12, "Compute"),
]
for name, flops, bytes_, bound in kernels:
    ai = flops / bytes_
    print(f"{name:<25} {flops:<12} {bytes_:<12} {ai:<10.3f} {bound}")

# T4 ridge point
t4_peak_gflops = 8100  # FP32
t4_peak_bw = 320       # GB/s
ridge = t4_peak_gflops / t4_peak_bw
print(f"\nT4 Ridge Point: {ridge:.1f} FLOPS/byte")
print(f"Kernels with AI < {ridge:.1f} are memory-bound")
print(f"Kernels with AI > {ridge:.1f} are compute-bound")

# ------------------------------------------------------------------
# Demo 2: Measure actual performance vs roofline
# ------------------------------------------------------------------
@cuda.jit
def vector_add(a, b, c, n):
    idx = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if idx < n:
        c[idx] = a[idx] + b[idx]

@cuda.jit
def saxpy(scalar, x, y, out, n):
    idx = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if idx < n:
        out[idx] = scalar * x[idx] + y[idx]

@cuda.jit
def heavy_compute(a, b, out, n):
    """Many FLOPs per element (high AI)."""
    idx = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if idx < n:
        val = a[idx]
        for _ in range(100):  # 100 iterations of 2 FLOPs = 200 FLOPs
            val = val * b[idx] + a[idx]  # FMA (1 multiply + 1 add)
        out[idx] = val

def bench(kernel, args, grid, block, warmup=10, iters=50):
    for _ in range(warmup):
        kernel[grid, block](*args)
    cuda.synchronize()
    start = time.perf_counter()
    for _ in range(iters):
        kernel[grid, block](*args)
    cuda.synchronize()
    return (time.perf_counter() - start) / iters * 1000

n = 4_000_000
TPB = 256
BPG = math.ceil(n / TPB)

a = cuda.to_device(np.random.randn(n).astype(np.float32))
b = cuda.to_device(np.random.randn(n).astype(np.float32))
c = cuda.device_array(n, dtype=np.float32)

print("\n=== Measured Performance vs Roofline ===")
print(f"{'Kernel':<20} {'Time (ms)':<12} {'AI (F/B)':<10} {'GFLOPS':<10} "
      f"{'Eff BW (GB/s)':<15} {'Bottleneck'}")
print("-" * 80)

# Vector add: 1 FLOP, 12 bytes
t = bench(vector_add, (a, b, c, n), BPG, TPB)
flops = n * 1
bytes_moved = n * 12
gflops = flops / (t / 1000) / 1e9
eff_bw = bytes_moved / (t / 1000) / 1e9
ai = 1 / 12
print(f"{'Vector add':<20} {t:<12.4f} {ai:<10.3f} {gflops:<10.2f} {eff_bw:<15.1f} {'Memory'}")

# SAXPY: 2 FLOPS, 12 bytes
t = bench(saxpy, (2.0, a, b, c, n), BPG, TPB)
flops = n * 2
bytes_moved = n * 12
gflops = flops / (t / 1000) / 1e9
eff_bw = bytes_moved / (t / 1000) / 1e9
ai = 2 / 12
print(f"{'SAXPY':<20} {t:<12.4f} {ai:<10.3f} {gflops:<10.2f} {eff_bw:<15.1f} {'Memory'}")

# Heavy compute: ~200 FLOPS, 12 bytes
t = bench(heavy_compute, (a, b, c, n), BPG, TPB)
flops = n * 200
bytes_moved = n * 12
gflops = flops / (t / 1000) / 1e9
eff_bw = bytes_moved / (t / 1000) / 1e9
ai = 200 / 12
print(f"{'Heavy compute':<20} {t:<12.4f} {ai:<10.3f} {gflops:<10.2f} {eff_bw:<15.1f} {'Compute'}")

# ------------------------------------------------------------------
# Demo 3: Kernel fusion benefit
# ------------------------------------------------------------------
@cuda.jit
def step1(a, b, temp, n):
    idx = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if idx < n:
        temp[idx] = a[idx] * b[idx]

@cuda.jit
def step2(temp, b, out, n):
    idx = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if idx < n:
        out[idx] = temp[idx] + b[idx]

@cuda.jit
def fused(a, b, out, n):
    idx = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if idx < n:
        out[idx] = a[idx] * b[idx] + b[idx]

temp = cuda.device_array(n, dtype=np.float32)

# Two-kernel approach
def bench_two_kernels():
    step1[BPG, TPB](a, b, temp, n)
    step2[BPG, TPB](temp, b, c, n)

for _ in range(10):
    bench_two_kernels()
cuda.synchronize()
start = time.perf_counter()
for _ in range(50):
    bench_two_kernels()
cuda.synchronize()
t_two = (time.perf_counter() - start) / 50 * 1000

t_fused = bench(fused, (a, b, c, n), BPG, TPB)

print(f"\n=== Kernel Fusion Benefit ===")
print(f"Two separate kernels: {t_two:.4f} ms")
print(f"Fused kernel:         {t_fused:.4f} ms")
print(f"Speedup from fusion:  {t_two / t_fused:.2f}x")
print(f"(Fusion eliminates intermediate global memory write + read)")

### Lesson example: Profiling & Optimization Loop

In [ ]:
!pip install numba

from numba import cuda
import numpy as np
import math
import time

# ==================================================================
# Complete profiling & optimization example
# Task: compute output[i] = sqrt(a[i]^2 + b[i]^2)
# ==================================================================

# --- Benchmark Harness ---
def benchmark(kernel, grid, block, args, warmup=10, iters=50):
    for _ in range(warmup):
        kernel[grid, block](*args)
    cuda.synchronize()
    times = []
    for _ in range(iters):
        cuda.synchronize()
        start = time.perf_counter()
        kernel[grid, block](*args)
        cuda.synchronize()
        t = (time.perf_counter() - start) * 1000
        times.append(t)
    times = np.array(times)
    return {'mean': np.mean(times), 'std': np.std(times),
            'min': np.min(times), 'median': np.median(times)}

def analyze(label, time_ms, n, flops_per_elem, bytes_per_elem):
    total_flops = n * flops_per_elem
    total_bytes = n * bytes_per_elem
    gflops = total_flops / (time_ms / 1000) / 1e9
    bw = total_bytes / (time_ms / 1000) / 1e9
    ai = flops_per_elem / bytes_per_elem
    print(f"{label:<24} {time_ms:>8.4f} ms | {bw:>7.1f} GB/s | {gflops:>7.1f} GFLOPS | AI={ai:.2f}")
    return time_ms

# --- Kernel versions ---
@cuda.jit
def magnitude_v1(a, b, output, n):
    """Naive: uses ** operator."""
    idx = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if idx < n:
        output[idx] = math.sqrt(a[idx] ** 2 + b[idx] ** 2)

@cuda.jit
def magnitude_v2(a, b, output, n):
    """Better: uses multiply instead of pow."""
    idx = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if idx < n:
        ai = a[idx]
        bi = b[idx]
        output[idx] = math.sqrt(ai * ai + bi * bi)

@cuda.jit
def magnitude_v3(a, b, output, n):
    """Best: multiple elements per thread for ILP."""
    base = (cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x) * 4
    for i in range(4):
        idx = base + i
        if idx < n:
            ai = a[idx]
            bi = b[idx]
            output[idx] = math.sqrt(ai * ai + bi * bi)

# --- Setup ---
n = 8_000_000
TPB = 256
BPG = math.ceil(n / TPB)
BPG_v3 = math.ceil(n / (TPB * 4))  # 4 elements per thread

a = cuda.to_device(np.random.randn(n).astype(np.float32))
b = cuda.to_device(np.random.randn(n).astype(np.float32))
out = cuda.device_array(n, dtype=np.float32)

print("=== GPU Kernel Optimization: Step by Step ===")
print(f"Problem: output[i] = sqrt(a[i]^2 + b[i]^2), N = {n:,}")
print(f"FLOPS/elem: 4 (mul, mul, add, sqrt), Bytes/elem: 12 (2 reads + 1 write)")
print(f"Arithmetic intensity: {4/12:.2f} FLOPS/byte (memory-bound!)\n")
print(f"{'Version':<24} {'Time':>12} | {'BW':>9} | {'Compute':>11} | {'AI'}")
print("-" * 75)

results = []
r1 = benchmark(magnitude_v1, BPG, TPB, (a, b, out, n))
t1 = analyze("V1: Naive (x**2)", r1['mean'], n, 4, 12)
results.append(('V1', t1))

# Verify correctness
out_host = out.copy_to_host()
a_host = a.copy_to_host()
b_host = b.copy_to_host()
expected = np.sqrt(a_host**2 + b_host**2)
assert np.allclose(out_host, expected, atol=1e-5), "V1 incorrect!"

r2 = benchmark(magnitude_v2, BPG, TPB, (a, b, out, n))
t2 = analyze("V2: x*x instead", r2['mean'], n, 4, 12)
results.append(('V2', t2))

r3 = benchmark(magnitude_v3, BPG_v3, TPB, (a, b, out, n))
t3 = analyze("V3: 4 elems/thread", r3['mean'], n, 4, 12)
results.append(('V3', t3))

# Verify V3
out_host = out.copy_to_host()
assert np.allclose(out_host, expected, atol=1e-5), "V3 incorrect!"

# NumPy reference
cuda.synchronize()
start = time.perf_counter()
np_result = np.sqrt(a_host**2 + b_host**2)
t_np = (time.perf_counter() - start) * 1000
print(f"{'NumPy (CPU)':<24} {t_np:>8.4f} ms |")

print(f"\n=== Improvement Summary ===")
for name, t in results:
    print(f"{name}: {t:.4f} ms ({t1/t:.2f}x vs V1, {t_np/t:.1f}x vs NumPy)")

# --- Demonstrate the importance of warmup ---
print("\n=== Why Warmup Matters ===")

@cuda.jit
def fresh_kernel(a, out, n):
    idx = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if idx < n:
        out[idx] = a[idx] * 2.0

# First call includes JIT compilation
cuda.synchronize()
start = time.perf_counter()
fresh_kernel[BPG, TPB](a, out, n)
cuda.synchronize()
t_first = (time.perf_counter() - start) * 1000

# Second call is warmed up
cuda.synchronize()
start = time.perf_counter()
fresh_kernel[BPG, TPB](a, out, n)
cuda.synchronize()
t_second = (time.perf_counter() - start) * 1000

print(f"First call (includes JIT):  {t_first:.2f} ms")
print(f"Second call (warmed up):    {t_second:.4f} ms")
print(f"Ratio: {t_first/t_second:.0f}x -- THIS is why warmup is essential!")

---

## Exercise: Profile and Optimize a Poorly-Written Kernel


In [ ]:
from numba import cuda
import numpy as np
import math
import time

# --- The deliberately BAD kernel ---
@cuda.jit
def kernel_naive(a, b, output, n, stride):
    """Poorly written kernel with multiple performance issues."""
    idx = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if idx < n:
        # Issue 1: Strided access (uncoalesced)
        real_idx = (idx * stride) % n
        # Issue 2: Using ** (compiled to slow pow())
        val_a_sq = a[real_idx] ** 2
        val_b_sq = b[real_idx] ** 2
        # Issue 3: Redundant global memory load
        log_val = math.log(abs(a[real_idx]) + 1.0)
        output[real_idx] = math.sqrt(val_a_sq + val_b_sq) + log_val

# TODO: Implement optimized versions
# @cuda.jit
# def kernel_v2(a, b, output, n):  # Fix coalescing

# @cuda.jit
# def kernel_v3(a, b, output, n):  # Fix operations

# @cuda.jit
# def kernel_v4(a, b, output, n):  # Multi-element per thread

def benchmark(kernel, grid, block, args, warmup=10, iters=50):
    """Time a kernel, return mean time in ms."""
    # TODO: implement with warmup and synchronization
    pass

# Setup
n = 4_000_000
TPB = 256
BPG = math.ceil(n / TPB)

# TODO: Create data, benchmark each version, verify correctness, print results table

## Your tasks

You are given a deliberately poorly-written kernel that computes `output[i] = sqrt(a[i]^2 + b[i]^2) + log(abs(a[i]) + 1)` for large arrays. Your task is to profile it, identify the bottlenecks, and apply optimizations step by step.

**Requirements:**
1. Start with the provided naive kernel and measure its performance
2. Calculate achieved bandwidth and GFLOPS to determine if memory-bound or compute-bound
3. Apply at least 3 optimizations, measuring after each one:
   - Fix the uncoalesced access pattern (the naive version reads from a transposed-like layout)
   - Replace expensive operations (pow, redundant loads)
   - Try processing multiple elements per thread
4. Report a summary table showing time, bandwidth, and speedup for each version
5. Verify correctness by comparing all versions against numpy

**Hints:**
- The naive kernel loads data with stride -- fixing this alone may give 3-10x speedup
- `x ** 2` compiles to a pow() call; `x * x` is a simple multiply (much faster)
- Loading `a[idx]` twice generates two separate memory transactions -- cache it in a local variable
- Process 2 or 4 elements per thread for instruction-level parallelism
- Use `cuda.synchronize()` before and after timing

When you're done, head back to the [lesson page](https://llm.thelittleone.rocks/#/module/cuda-performance) for the solution and discussion.